In [1]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd
from numpy import nan
from functions import functions
import json

from urllib.parse import quote_plus
from sqlalchemy import create_engine
engine = create_engine('postgresql://zbrothers:%s@localhost:5434/zbrothers' % quote_plus('zbrothers@2026'))

In [42]:
PRODUCT_PICKING_COLUMNS = ['id', 'picking_operator_id', 'idOrigem', 'objOrigem', 'situacao',
       'situacaoCheckout', 'dataCriacao', 'itens', 'qtdVolumes', 'numero',
       'dataEmissao', 'numeroPedidoEcommerce', 'idFormaEnvio', 'formaEnvio',
       'idContato', 'destinatario', 'situacaoOrigem', 'dataSeparacao',
       'dataCheckout', 'idOrigemVinc', 'objOrigemVinc', 'situacaoVenda']

In [48]:
df_0 = pd.read_parquet('data/product_picking-2026-04-29 15:25:09.parquet').astype({'id': int})

In [51]:
df_0['dataCriacao'].unique()

<DatetimeArray>
['2026-04-29 00:00:00']
Length: 1, dtype: datetime64[us]

In [52]:
df_2['dataCriacao'].unique()

<DatetimeArray>
['2026-04-30 00:00:00']
Length: 1, dtype: datetime64[us]

In [4]:
df_users = pd.DataFrame(columns=['id', 'name'], data=[
    [1079022993, 'Luan Daniel'],
    [1240202884, 'Henrique'],
    [879165082, 'Gustavo'],
    [1084497214, 'Gabriel Augusto']
])

In [ ]:
# df_users.to_sql("picking_operators", engine, if_exists='append', index=False)

4

In [54]:
df_6 = pd.read_parquet('data/product_picking-2026-04-30 16:05:44.parquet')
df_6.to_sql('product_picking', engine, if_exists='append', index=False)

371

In [ ]:
# df_1 = pd.read_parquet('data/product_picking-2026-04-30 10:05:31.parquet')
# df_1.to_sql('product_picking', engine, if_exists='append', index=False)

344

In [19]:
df_1_db = pd.read_sql('SELECT * FROM product_picking', engine).astype({'id': int})

In [20]:
df_2 = pd.read_parquet('data/product_picking-2026-04-30 11:05:17.parquet').astype({'id': int})

In [23]:
df_merge = df_2.merge(df_1_db, on='id', how='left', validate='1:1', suffixes=('_new', '_db'))

In [32]:
df_merge_new = df_merge[df_merge['idOrigem_db'].isna()]
df_merge_diff = df_merge[~df_merge['idOrigem_db'].isna()]

if df_merge_new.shape[0] + df_merge_diff.shape[0] != df_merge.shape[0]:
    raise Exception('Diferença nas quantidade de linhas')

In [43]:
df_1_db_select = df_1_db[df_1_db['id'].isin(df_merge_diff['id'])].sort_values(['id'], ascending=True).reset_index(drop=True)[PRODUCT_PICKING_COLUMNS]
df_2_select = df_2[df_2['id'].isin(df_merge_diff['id'])].sort_values(['id'], ascending=True).reset_index(drop=True)[PRODUCT_PICKING_COLUMNS]

In [46]:
df_2_select.compare(df_1_db_select, keep_equal=False)

picking_operator_id       situacao       situacaoCheckout        \
                   self other     self other             self other   
0                   NaN   NaN      NaN   NaN              NaN   NaN   
1            1240202884   NaN        3     2                2     1   
2                   NaN   NaN      NaN   NaN              NaN   NaN   
3             879165082   NaN        3     2                2     1   
4                   NaN   NaN      NaN   NaN              NaN   NaN   
..                  ...   ...      ...   ...              ...   ...   
339                 NaN   NaN      NaN   NaN              NaN   NaN   
340                 NaN   NaN      NaN   NaN              NaN   NaN   
341                 NaN   NaN      NaN   NaN              NaN   NaN   
342                 NaN   NaN      NaN   NaN              NaN   NaN   
343                 NaN   NaN      NaN   NaN              NaN   NaN   

                                                 itens  \
                                                  self   
0    [{"idProduto": "1069373941", "descricao": "Sup...   
1    [{"idProduto": "1070980833", "descricao": "Esc...   
2    [{"idProduto": "1061109227", "descricao": "BAN...   
3    [{"idProduto": "1055066076", "descricao": "Ali...   
4    [{"idProduto": "908324222", "descricao": "ARAM...   
..                                                 ...   
339  [{"idProduto": "847488354", "descricao": "ESCO...   
340  [{"idProduto": "832411846", "descricao": "LIMP...   
341  [{"idProduto": "1072666697", "descricao": "Sop...   
342  [{"idProduto": "1055547254", "descricao": "AFI...   
343  [{"idProduto": "1074066770", "descricao": "LAV...   

                                                       qtdVolumes        \
                                                 other       self other   
0    [{'codigo': '00013073.6', 'unidade': 'UN', 'de...          0     0   
1    [{'codigo': '6325008200', 'unidade': 'PC', 'de...          0     0   
2    [{'codigo': '3540036004', 'unidade': 'UN', 'de...          0     0   
3    [{'codigo': '3662061505', 'unidade': 'UN', 'de...          0     0   
4    [{'codigo': '7432008001', 'unidade': 'KG', 'de...          0     0   
..                                                 ...        ...   ...   
339  [{'codigo': '6364036075', 'unidade': 'PC', 'de...          0     0   
340  [{'codigo': '6510201000', 'unidade': 'PC', 'de...          0     0   
341  [{'codigo': '00036058.2', 'unidade': 'UN', 'de...          0     0   
342  [{'codigo': '6001095127', 'unidade': 'UN', 'de...          0     0   
343  [{'codigo': '71000562', 'unidade': 'PC', 'desc...          0     0   

    dataCheckout        
            self other  
0            NaT   NaN  
1     2026-04-30  None  
2            NaT   NaN  
3     2026-04-30  None  
4            NaT   NaN  
..           ...   ...  
339          NaT   NaN  
340          NaT   NaN  
341          NaT   NaN  
342          NaT   NaN  
343          NaT   NaN  

[344 rows x 12 columns]

In [40]:
df_2_select

,id,idOrigem,objOrigem,situacao,situacaoCheckout,dataCriacao,itens,qtdVolumes,numero,dataEmissao,...,formaEnvio,idContato,destinatario,situacaoOrigem,dataSeparacao,picking_operator_id,dataCheckout,idOrigemVinc,objOrigemVinc,situacaoVenda
0,1075596875,1075596867,notafiscal,2,1,2026-04-30,"[{""idProduto"": ""1069373941"", ""descricao"": ""Sup...",0,189551,2026-04-30,...,Mercado Envios,1031488551,Edivaldo Kara,None,2026-04-30,NaN,NaT,1075593370,venda,6
1,1075596885,1075596876,notafiscal,3,2,2026-04-30,"[{""idProduto"": ""1070980833"", ""descricao"": ""Esc...",0,189553,2026-04-30,...,Mercado Envios,1031488574,Danilo Menelli,None,2026-04-30,1240202884,2026-04-30,1075593502,venda,6
2,1075596886,1075596869,notafiscal,2,1,2026-04-30,"[{""idProduto"": ""1061109227"", ""descricao"": ""BAN...",0,189552,2026-04-30,...,Mercado Envios,1031488571,Joao Guilherme Boica Schultz,None,2026-04-30,NaN,NaT,1075593494,venda,6
3,1075596901,1075596887,notafiscal,3,2,2026-04-30,"[{""idProduto"": ""1055066076"", ""descricao"": ""Ali...",0,189554,2026-04-30,...,Mercado Envios,1031488582,Jose Antonio Raduan Alba,None,2026-04-30,879165082,2026-04-30,1075593534,venda,6
4,1075596911,1075596894,notafiscal,2,1,2026-04-30,"[{""idProduto"": ""908324222"", ""descricao"": ""ARAM...",0,189555,2026-04-30,...,Mercado Envios,1031488584,Kleber Rodrigues Braga,None,2026-04-30,NaN,NaT,1075593552,venda,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,1075602335,1075602323,notafiscal,1,1,2026-04-30,"[{""idProduto"": ""847488354"", ""descricao"": ""ESCO...",0,189896,2026-04-30,...,Mercado Envios,1031489405,Alex Sandro de Souza Pereira,None,NaT,NaN,NaT,1075601854,venda,6
340,1075602343,1075602336,notafiscal,1,1,2026-04-30,"[{""idProduto"": ""832411846"", ""descricao"": ""LIMP...",0,189897,2026-04-30,...,Mercado Envios,1031489403,Marcos Vinicios Coutinho Sangineto,None,NaT,NaN,NaT,1075601850,venda,6
341,1075602354,1075602345,notafiscal,1,1,2026-04-30,"[{""idProduto"": ""1072666697"", ""descricao"": ""Sop...",0,189898,2026-04-30,...,Mercado Envios,1031489402,Jorge Alberto Rosa da Silva,None,NaT,NaN,NaT,1075601849,venda,6
342,1075602612,1075602605,notafiscal,1,1,2026-04-30,"[{""idProduto"": ""1055547254"", ""descricao"": ""AFI...",0,189899,2026-04-30,...,Mercado Envios,1031489460,Nelson Constantino Conceicao Junior,None,NaT,NaN,NaT,1075602531,venda,6


In [41]:
df_1_db_select.columns

Index(['id', 'picking_operator_id', 'idOrigem', 'objOrigem', 'situacao',
       'situacaoCheckout', 'dataCriacao', 'itens', 'qtdVolumes', 'numero',
       'dataEmissao', 'numeroPedidoEcommerce', 'idFormaEnvio', 'formaEnvio',
       'idContato', 'destinatario', 'situacaoOrigem', 'dataSeparacao',
       'dataCheckout', 'idOrigemVinc', 'objOrigemVinc', 'situacaoVenda'],
      dtype='str')